In [ ]:
import os
import copy
import configparser
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import Dataset, random_split
from clearml import Task

# -------------------- ClearML -------------------- #
task = Task.init(project_name="PitchSense", task_name="PitchSenseMergedDataset")
task.set_base_task(None)

# -------------------- Dataset -------------------- #
class PitchSenseDataset(Dataset):
    def __init__(self, root_dirs):
        """
        root_dirs: list of Paths (train + test)
        """
        self.samples = []
        for root in root_dirs:
            root = Path(root)
            if not root.exists():
                continue
            for match_id in sorted(os.listdir(root)):
                match_path = root / match_id
                if not match_path.is_dir():
                    continue
                sample = self._process_match(match_path)
                if sample:
                    self.samples.append(sample)

    def _read_ini_to_dict(self, path: Path) -> dict:
        parser = configparser.ConfigParser()
        parser.read(path)
        return {section: dict(parser[section]) for section in parser.sections()}

    def _get_inis(self, tree):
        game_config, seq_config = {}, {}
        for sub_path in tree:
            if sub_path.is_file():
                if sub_path.stem == "gameinfo":
                    game_config = self._read_ini_to_dict(sub_path).get("Sequence", {})
                elif sub_path.stem == "seqinfo":
                    seq_config = self._read_ini_to_dict(sub_path).get("Sequence", {})
        return game_config, seq_config

    def _get_mapper(self, match_config):
        tracklet_map = {}
        for key, value in match_config.items():  
            if key.startswith("trackletid_"):
                try:
                    idx = int(key.split("_")[1])
                    name = value.split(";")[0]  
                    tracklet_map[idx] = name
                except Exception:
                    continue
        return tracklet_map

    def _boiler_plate(self, tree, match_config):
        gt = det = pd.DataFrame()
        tree_copy = copy.copy(tree)
        for item in tree_copy:
            if item.stem == "gt" and item.is_dir():
                gt_file = item / "gt.txt"
                if gt_file.exists():
                    gt = pd.read_csv(gt_file, header=None)
                tree.remove(item)
            elif item.stem == "det" and item.is_dir():
                det_file = item / "det.txt"
                if det_file.exists():
                    det = pd.read_csv(det_file, header=None)
                tree.remove(item)
        if not gt.empty:
            gt.columns = ['frame', 'track_id', 'x', 'y', 'w', 'h', 'class_id', 'f1','f2','f3']
        if not det.empty:
            det.columns = ['frame', 'track_id', 'x', 'y', 'w', 'h', 'class_id', 'f1','f2','f3']

        tracklet_map = self._get_mapper(match_config)
        if not gt.empty:
            gt['name'] = gt['track_id'].map(tracklet_map).fillna("unknown")
        if not det.empty:
            det['name'] = det['track_id'].map(tracklet_map).fillna("unknown")
        return gt, det, tree

    def _process_match(self, match_path: Path):
        try:
            tree = [f for f in match_path.iterdir()]
            game_config, seq_config = self._get_inis(tree)
            match_config = {**game_config, **seq_config}
            tree_dirs = [f for f in match_path.iterdir() if f.is_dir()]
            gt, det, _ = self._boiler_plate(tree_dirs, match_config)
            if gt.empty and det.empty:
                return None
            return {
                "match_id": match_path.name,
                "gt": gt,
                "det": det,
                "config": match_config
            }
        except Exception as e:
            print(f"Warning: failed to process {match_path}: {e}")
            return None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# -------------------- Merge train + test -------------------- #
train_root = "/path/to/train"
test_root = "/path/to/test"

dataset = PitchSenseDataset([train_root, test_root])

# -------------------- Train/Val split -------------------- #
val_ratio = 0.15
val_size = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# -------------------- ClearML upload -------------------- #
task.upload_artifact(name="train_dataset", artifact_object=train_dataset)
task.upload_artifact(name="val_dataset", artifact_object=val_dataset)
task.close()

print(f"Dataset samples: {len(dataset)}")
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")